# IDAP Alarms — Finding the *Predictive* Ones (Prescriptive already labeled)

**The setup.** Some alarms are **already labeled `PRESCRIPTIVE`** — a person tagged them by hand, each with a note in `what_fixed_it` saying how it was fixed. Those are done; we leave them alone. This notebook only looks at the alarms that **aren't labeled yet**, and asks one question about each:

> Is this alarm **PREDICTIVE** — does it break on a *steady, repeating schedule* we could see coming and get ahead of — **or does the data we have simply not tell us?**

**Being honest up front.** "Predictive" really just means *its timing is regular enough to forecast* — the time between one firing and the next is fairly even, not random. To measure that properly you need the **time of every single firing**. This file only gives us **summary numbers**: a total count, the first and last time it fired, how many days it was active, and how many repair tickets it caused. So we do two separate things:

1. **Squeeze out every bit of signal the summary numbers allow** — with a full set of statistical tests and models — and sort the unlabeled alarms into **PREDICTIVE-CANDIDATE** or **UNCLASSIFIABLE** (with a reason why).
2. **Say clearly where the data runs out.** The last section gives a straight answer on whether the data is good enough, and lists exactly what we'd need (sensor/IoT data, repair-outcome data, the raw event log, …) to turn a *candidate* into a *confirmed* predictive alarm.

**IMPORTANT**: *We have no record of whether past fixes actually worked, so "confidence" in this notebook means a label is **steady** — it doesn't flip when we redraw the lines slightly — **not** that it's been proven right.*

## 0 · Setup & the measures we build

We load the data and turn the raw columns into a few simple measures. Every one is built from **summary numbers only** — we say so plainly, because it's the whole crux of what we can and can't conclude.

- **Active-day ratio** — out of all the days an alarm was "around," on how many did it actually fire? $\;\text{ADR}=\dfrac{\text{distinct active days}}{\text{window length (days)}}\in[0,1]$. Close to 1 = it keeps happening, spread evenly over time (looks forecastable); close to 0 = it all happened in a few days and then went quiet (a burst). *This is the best "regularity" hint the summary numbers can give us.*
- **Events per active day** — on the days it did fire, how many times on average = firings ÷ active days.
- **Mean gap (days)** — rough average time between firings = window ÷ firings (the flip side of the active-day ratio).
- **CM per occurrence** — repair tickets ÷ firings — how much fixing each firing tends to cause.
- **Downtime per occurrence** — total downtime ÷ firings.

**Why these.** They pull apart things the raw counts lump together: the *active-day ratio* separates "spread evenly over time" (what makes a failure forecastable) from "just fires a lot"; *events/day* and *mean gap* describe how intense and how spaced-out it is; *CM per occurrence* measures repair effort fairly no matter how often the alarm fires. What we **don't** have — and can't invent — is the time of each individual firing, which is what you'd need to measure true regularity.

In [ ]:
import os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_selection import mutual_info_classif
warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams.update({"figure.dpi":110,"axes.grid":True,"grid.alpha":.25})
pd.set_option("display.width",200,"display.max_columns",40)

CANDIDATES=[r"/mnt/user-data/uploads/Cur_data.xlsx", "Cur_data.xlsx",
            r"C:\Users\L138397\Downloads\prac\Cur_data.xlsx"]
DATA_PATH=next((p for p in CANDIDATES if os.path.exists(p)), CANDIDATES[0])
OUT="/mnt/user-data/outputs" if os.path.isdir("/mnt/user-data/outputs") else "."
print("using:", DATA_PATH)

In [ ]:
df = pd.read_excel(DATA_PATH)
for c in ["first_occurrence","last_occurrence"]:
    df[c]=pd.to_datetime(df[c],errors="coerce")

span=((df.last_occurrence-df.first_occurrence).dt.total_seconds()/86400).clip(lower=1)
df["span_days"]            = span
df["events_per_active_day"]= df.occurrence_count/df.distinct_days_affected.clip(lower=1)
df["active_day_ratio"]     = (df.distinct_days_affected/span).clip(0,1)      # main regularity proxy
df["mean_gap_days"]        = span/df.occurrence_count.clip(lower=1)
df["cm_per_occ"]           = df.maximo_cm_count/df.occurrence_count.clip(lower=1)
df["downtime_per_occ"]     = df.total_downtime_mins/df.occurrence_count.clip(lower=1)

# equipment type parsed from the Maximo location — an INDEPENDENT signal, used only for validation
KW=["CARTON","FEEDER","LABEL","MODULE","PALLET","CONV","VISION","CASE","ROBOT","PLC","CAP","GLUE","DSSA","QOB"]
df["equip_type"]=df.top_asset_location.apply(
    lambda s: next((k for k in KW if k in str(s).upper()),"OTHER") if pd.notna(s) else "UNKNOWN")

presc = df.classification.eq("PRESCRIPTIVE")   # GIVEN — left untouched
unlab = df.classification.isna()               # the population we must decide on

# --- data inventory: what do we actually have? (this drives the final verdict) ---
have = {
 "per-event timestamp log (needed for true inter-arrival regularity)": False,
 "aggregate occurrence_count": True,
 "first/last occurrence only (window, not the sequence)": True,
 "distinct active days": True,
 "corrective WO count (maximo_cm_count)": True,
 "preventive/planned WO count (PM)": any("pm" in c.lower() for c in df.columns),
 "condition / IoT sensor streams (vibration, current, temp, …)": False,
 "maintenance OUTCOME / results (did failure recur after fix?)": False,
}
print(f"rows: {len(df)}   |   PRESCRIPTIVE (given): {presc.sum()}   |   unlabeled to decide: {unlab.sum()}")
print(f"window: {df.first_occurrence.min().date()} -> {df.last_occurrence.max().date()}  (median span {span.median():.0f} d)\n")
print("DATA INVENTORY")
for k,v in have.items(): print(f"  [{'YES' if v else 'NO '}]  {k}")

## 1 · Are the measures shaped like a normal bell curve?  (Shapiro–Wilk)

**What we're checking.** Whether each measure is roughly a normal "bell curve" shape. This decides which correlation is fair to use later: bell-shaped → the ordinary (Pearson) correlation; lopsided → the rank-based (Spearman) one.

**The test.** $\;W=\dfrac{\left(\sum_i a_i x_{(i)}\right)^2}{\sum_i (x_i-\bar x)^2}$ — $W$ compares your sorted values against a perfect bell curve ($x_{(i)}$ = your values sorted low-to-high, $\bar x$ = their average, $a_i$ = fixed bell-curve reference weights). $W$ near 1 = looks normal. It also gives a **p-value**; the starting assumption is "the data is normal," and **p < 0.05 means we reject that** — it's not normal.

**Why bother.** It's the standard normality check, and it tells us which correlation we're allowed to trust.

In [ ]:
FEAT=["occurrence_count","total_downtime_mins","avg_downtime_mins","distinct_days_affected",
      "span_days","events_per_active_day","active_day_ratio","mean_gap_days","cm_per_occ","downtime_per_occ"]
rows=[]
for c in FEAT:
    x=df[c].replace([np.inf,-np.inf],np.nan).dropna()
    x=x.sample(4000,random_state=0) if len(x)>4000 else x
    W,p=stats.shapiro(x)
    rows.append({"feature":c,"W":round(W,3),"p":f"{p:.1e}",
                 "decision":"REJECT H0 (non-normal)" if p<0.05 else "fail to reject"})
print(pd.DataFrame(rows).to_string(index=False))
print("\n=> every feature rejects normality -> use Spearman throughout.")

fig,axes=plt.subplots(2,5,figsize=(15,6))
for a,c in zip(axes.ravel(),FEAT):
    a.hist(np.log1p(df[c].replace([np.inf,-np.inf],np.nan).clip(lower=0).dropna()),bins=40,color="#3182bd")
    a.set_title(f"log1p({c})",fontsize=8); a.tick_params(labelsize=6)
fig.suptitle("Feature distributions (log scale) — long right tails confirm non-normality",weight="bold")
fig.tight_layout(); plt.show()